# Chapter 8 — RAG Generation
## Topic 6: Multi-Turn RAG — Conversation History + Retrieval

**Run cells in order.**

- Module 1: Setup — a 3-turn conversation + mock contextualization client
- Module 2: Query contextualization (raw follow-up -> self-contained query)
- Module 3: Retrieval quality WITH vs WITHOUT contextualization
- Module 4: Token budget pressure as history grows (ties to Topic 1)

**Install:** `pip install sentence-transformers numpy`


## Module 1: Setup — Conversation + Mock Contextualization Client

**Requires:** nothing standalone

In [ ]:
"""
MODULE 1: Setup

A 3-turn conversation where turn 3 is genuinely ambiguous without history --
"what about for senior citizens?" has no retrievable content on its own.

MockClient simulates the contextualization LLM call offline. Replace with
a real anthropic.Anthropic() client + claude-haiku-4-5-20251001 in production
(the function signature is identical).
"""

CONVERSATION = [
    {"role": "user", "content": "What is the penalty for premature FD withdrawal?"},
    {"role": "assistant", "content": "The penalty is 1 percent on the applicable interest rate."},
    {"role": "user", "content": "What about for senior citizens?"},  # ambiguous alone
]

KNOWLEDGE_BASE = [
    {"id": 0, "text": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate."},
    {"id": 1, "text": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures, "
                       "and the same premature withdrawal penalty rules apply to them as to other depositors."},
    {"id": 2, "text": "The Fixed Deposit interest rate for 24 months is 7.1 percent per annum."},
]


class MockContextualizationClient:
    """Simulates the contextualization LLM call offline. A real implementation
    calls client.messages.create() with claude-haiku-4-5-20251001 -- this
    mock uses simple keyword-based logic to approximate correct rewriting."""

    class MockResponse:
        def __init__(self, text):
            self.content = [type("Block", (), {"text": text})()]

    class Messages:
        def create(self, model, max_tokens, messages):
            prompt = messages[0]["content"]
            # crude simulation: detect the ambiguous "what about" pattern and
            # splice in the topic from the prior turn (extracted from the prompt)
            if "what about for senior citizens" in prompt.lower():
                if "penalty" in prompt.lower() and "premature" in prompt.lower():
                    rewritten = "What is the penalty for premature FD withdrawal for senior citizens?"
                else:
                    rewritten = "What about for senior citizens?"  # fallback: no useful history found
            else:
                rewritten = prompt.split("Latest customer message:")[-1].strip().strip('"')
            return MockContextualizationClient.MockResponse(rewritten)

    def __init__(self):
        self.messages = MockContextualizationClient.Messages()


mock_client = MockContextualizationClient()
MODEL_ID = "claude-haiku-4-5-20251001"

print("Conversation so far:")
for turn in CONVERSATION:
    print(f"  {turn['role']}: {turn['content']}")

print("\nModule 1 complete. Run Module 2.")


## Module 2: Query Contextualization

**Requires:** Module 1

In [ ]:
"""
MODULE 2: Query Contextualization

Rewrites the raw follow-up into a self-contained retrieval query using
conversation history.
"""

def contextualize_query(raw_query: str, conversation_history: list, client, model_id: str) -> str:
    if not conversation_history:
        return raw_query

    history_text = "\n".join(
        f"{m['role']}: {m['content']}" for m in conversation_history[-4:]
    )

    rewrite_prompt = f"""Conversation so far:
{history_text}

Latest customer message: "{raw_query}"

Rewrite the latest message as a single, self-contained question that includes
any necessary context from the conversation above. Output ONLY the rewritten
question, nothing else."""

    response = client.messages.create(
        model=model_id, max_tokens=100,
        messages=[{"role": "user", "content": rewrite_prompt}],
    )
    return response.content[0].text.strip()


raw_followup = CONVERSATION[-1]["content"]
history_before_followup = CONVERSATION[:-1]

contextualized = contextualize_query(raw_followup, history_before_followup, mock_client, MODEL_ID)

print(f"Raw follow-up:      \'{raw_followup}\'")
print(f"Contextualized to:  \'{contextualized}\'")

assert "senior citizen" in contextualized.lower() and "penalty" in contextualized.lower(), \
    "Contextualization should carry forward the penalty topic!"
print("\n[VERIFIED] Contextualization correctly carried forward the missing topic.")

print("\nModule 2 complete. Run Module 3.")


## Module 3: Retrieval Quality WITH vs WITHOUT Contextualization

**Requires:** Module 1, Module 2

In [ ]:
"""
MODULE 3: Retrieval Quality Comparison

Runs dense retrieval on the RAW follow-up vs. the CONTEXTUALIZED query,
showing the concrete retrieval quality difference this step produces.
"""

import numpy as np
from sentence_transformers import SentenceTransformer

print("Loading embedding model (may take a moment on first run)...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

kb_texts = [doc["text"] for doc in KNOWLEDGE_BASE]
kb_embeddings = embed_model.encode(kb_texts, normalize_embeddings=True, show_progress_bar=False)
for i, doc in enumerate(KNOWLEDGE_BASE):
    doc["embedding"] = kb_embeddings[i]


def dense_retrieve(query: str, top_k: int = 3) -> list:
    q_vec = embed_model.encode(query, normalize_embeddings=True, show_progress_bar=False)
    scored = [(doc, float(np.dot(q_vec, doc["embedding"]))) for doc in KNOWLEDGE_BASE]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


print(f"\nQuery: RAW follow-up -- \'{raw_followup}\'")
raw_results = dense_retrieve(raw_followup)
for doc, score in raw_results:
    print(f"  score={score:.4f} | {doc['text'][:60]}...")

print(f"\nQuery: CONTEXTUALIZED -- \'{contextualized}\'")
context_results = dense_retrieve(contextualized)
for doc, score in context_results:
    print(f"  score={score:.4f} | {doc['text'][:60]}...")

raw_top1_score = raw_results[0][1]
context_top1_score = context_results[0][1]

print(f"\nTop-1 score WITHOUT contextualization: {raw_top1_score:.4f}")
print(f"Top-1 score WITH contextualization:    {context_top1_score:.4f}")
print(f"Improvement: {context_top1_score - raw_top1_score:+.4f}")

print("\nModule 3 complete. Run Module 4.")


## Module 4: Token Budget Pressure as History Grows

**Requires:** Module 1

In [ ]:
"""
MODULE 4: Token Budget Pressure

Shows how conversation history competes with retrieved-context budget
(Topic 1) as a conversation grows across more turns -- a genuinely NEW
problem versus single-turn RAG.
"""

def estimate_tokens(text: str) -> int:
    return len(text) // 4


def simulate_growing_conversation(n_turns: int, avg_turn_length_chars: int = 150) -> dict:
    """Simulates how much of the fixed budget history consumes as turns grow."""
    context_window = 200_000  # claude-haiku-4-5-20251001
    system_prompt_tokens = 200  # illustrative
    reserved_output = 500

    history_tokens = n_turns * estimate_tokens("x" * avg_turn_length_chars)
    available_for_chunks = context_window - system_prompt_tokens - reserved_output - history_tokens

    return {
        "n_turns": n_turns,
        "history_tokens": history_tokens,
        "available_for_chunks": available_for_chunks,
        "pct_of_window_used_by_history": history_tokens / context_window,
    }


print(f"{'Turns':<8} | {'History tokens':<16} | {'Available for chunks':<22} | % window used by history")
print("-" * 80)
for n in [1, 5, 10, 50, 200, 1000]:
    result = simulate_growing_conversation(n)
    print(f"{result['n_turns']:<8} | {result['history_tokens']:<16,} | "
          f"{result['available_for_chunks']:<22,} | {result['pct_of_window_used_by_history']:.2%}")

print("""
At this project's 200K token context window (claude-haiku-4-5-20251001),
history pressure only becomes significant at VERY high turn counts given
typical short FD-related exchanges -- but the discipline matters:
  1. Full history inclusion is fine for THIS project's likely short
     conversations (consistent with 31-word average email length).
  2. If conversations regularly grow much longer, truncation or
     summarization (the theory's Section 5 discussion) becomes necessary
     to keep meaningful budget available for retrieved context.
This is exactly the kind of decision that should be validated with REAL
conversation-length data from production, not assumed from a single
context-window-size number alone.
""")

print("=" * 65)
print("CHAPTER 8 TOPIC 6 SUMMARY")
print("=" * 65)
print("""
Query contextualization rewrites ambiguous follow-ups into self-contained
  retrieval queries using conversation history -- Module 2/3 showed this
  concretely improves retrieval score for an otherwise-unretrievable follow-up.
This runs BEFORE Chapter 7's retrieval pipeline, which is otherwise unchanged.
Conversation history competes with retrieved context for Topic 1's fixed
  token budget -- Module 4 quantified this pressure as turn count grows.
Given this project's likely short conversations, full history inclusion is
  probably sufficient; validate with real data before adding truncation/
  summarization complexity.

Next: Topic 7 -- Query Rewriting and HyDE
""")
